In [ ]:
# CELL 1 — Imports
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries loaded.")

# CELL 2 — Upload Dataset
df = pd.read_csv('../data/df_merged.csv')
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")

# CELL 3 — Data Preparation
# Convert IP to billions for readability
df['ind_prod_const_sa_billions'] = df['ind_prod_const_sa'] / 1e9

# Convert US imports to thousands for readability
df['usa_imp_sa_thousands'] = df['usa_imp_sa'] / 1e3

# Create interaction term
df['usa_imp_sa_thousands:exports_gdp_pct'] = df['usa_imp_sa_thousands'] * df['exports_gdp_pct']

print("✅ Variables prepared:")
print(f"   Dependent: ind_prod_const_sa_billions (IP in billions)")
print(f"   Independent:")
print(f"     - usa_imp_sa_thousands (US Imports in thousands)")
print(f"     - exports_gdp_pct (Exports/GDP %)")
print(f"     - usa_imp_sa_thousands:exports_gdp_pct (Interaction)")
print(f"     - cpi_yoy_nsa (CPI YoY %)")
print(f"     - exchnage_rate_vs_usd (Exchange Rate)")
print(f"\n   Observations: {len(df)}")
print(f"   Countries: {df['country'].unique().tolist()}")


# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1: FULL MULTIVARIATE OLS 
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  MODEL 1: FULL MULTIVARIATE OLS (Pooled, all countries)")
print("  IP (billions) = α + β₁(US_Imp) + β₂(Exp/GDP) + β₃(US_Imp × Exp/GDP)")
print("                + β₄(CPI) + β₅(Exchange Rate) + ε")
print("=" * 70)

# Using statsmodels formula API for clean output
model_full = smf.ols(
    'ind_prod_const_sa_billions ~ usa_imp_sa_thousands + exports_gdp_pct '
    '+ Q("usa_imp_sa_thousands:exports_gdp_pct") + cpi_yoy_nsa + exchnage_rate_vs_usd',
    data=df
).fit()

print(model_full.summary())


# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2: SAME MODEL — using add_constant 
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  MODEL 2: SAME MODEL (manual approach for verification)")
print("=" * 70)

X_vars = ['usa_imp_sa_thousands', 'exports_gdp_pct',
          'usa_imp_sa_thousands:exports_gdp_pct',
          'cpi_yoy_nsa', 'exchnage_rate_vs_usd']

X = sm.add_constant(df[X_vars])
y = df['ind_prod_const_sa_billions']

model_manual = sm.OLS(y, X).fit()
print(model_manual.summary())


# ══════════════════════════════════════════════════════════════════════════════
# MODEL 3: COUNTRY-LEVEL REGRESSIONS (same specification per country)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  MODEL 3: COUNTRY-LEVEL REGRESSIONS")
print("=" * 70)

country_models = {}
for country in sorted(df['country'].unique()):
    sub = df[df['country'] == country].copy()

    X_c = sm.add_constant(sub[['usa_imp_sa_thousands', 'exports_gdp_pct',
                                'cpi_yoy_nsa', 'exchnage_rate_vs_usd']])
    y_c = sub['ind_prod_const_sa_billions']

    model_c = sm.OLS(y_c, X_c).fit()
    country_models[country] = model_c

    print(f"\n{'─' * 70}")
    print(f"  {country.upper()}")
    print(f"{'─' * 70}")
    print(model_c.summary())


# ══════════════════════════════════════════════════════════════════════════════
# MODEL 4: PROGRESSIVE MODEL BUILDING 
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  MODEL 4: PROGRESSIVE MODEL BUILDING")
print("=" * 70)

# Model 4a: US Imports only
X4a = sm.add_constant(df[['usa_imp_sa_thousands']])
m4a = sm.OLS(y, X4a).fit()

# Model 4b: + Trade Dependency
X4b = sm.add_constant(df[['usa_imp_sa_thousands', 'exports_gdp_pct']])
m4b = sm.OLS(y, X4b).fit()

# Model 4c: + Interaction
X4c = sm.add_constant(df[['usa_imp_sa_thousands', 'exports_gdp_pct',
                           'usa_imp_sa_thousands:exports_gdp_pct']])
m4c = sm.OLS(y, X4c).fit()

# Model 4d: + Controls (CPI + Exchange Rate) — FULL MODEL
X4d = sm.add_constant(df[['usa_imp_sa_thousands', 'exports_gdp_pct',
                           'usa_imp_sa_thousands:exports_gdp_pct',
                           'cpi_yoy_nsa', 'exchnage_rate_vs_usd']])
m4d = sm.OLS(y, X4d).fit()

print(f"\n{'Model':<45} {'Adj. R²':>10} {'F-stat':>10} {'N':>6}")
print("─" * 75)
print(f"{'4a: US Imports only':<45} {m4a.rsquared_adj:>10.4f} {m4a.fvalue:>10.2f} {int(m4a.nobs):>6}")
print(f"{'4b: + Exports/GDP':<45} {m4b.rsquared_adj:>10.4f} {m4b.fvalue:>10.2f} {int(m4b.nobs):>6}")
print(f"{'4c: + Interaction (US × Trade Dep)':<45} {m4c.rsquared_adj:>10.4f} {m4c.fvalue:>10.2f} {int(m4c.nobs):>6}")
print(f"{'4d: + Controls (CPI + Exchange Rate)':<45} {m4d.rsquared_adj:>10.4f} {m4d.fvalue:>10.2f} {int(m4d.nobs):>6}")
print("─" * 75)

# Full summaries for each
for label, m in [("4a", m4a), ("4b", m4b), ("4c", m4c), ("4d", m4d)]:
    print(f"\n{'─' * 70}")
    print(f"  MODEL {label}")
    print(f"{'─' * 70}")
    print(m.summary())


# ══════════════════════════════════════════════════════════════════════════════
# ACADEMIC COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  ACADEMIC COMPARISON TABLE")
print("=" * 70)

all_vars = ['const', 'usa_imp_sa_thousands', 'exports_gdp_pct',
            'usa_imp_sa_thousands:exports_gdp_pct',
            'cpi_yoy_nsa', 'exchnage_rate_vs_usd']
all_labels = ['Constant', 'US Imports (thousands)', 'Exports/GDP (%)',
              'US Imports × Exports/GDP', 'CPI YoY (%)', 'Exchange Rate vs USD']

prog_models = [m4a, m4b, m4c, m4d]
prog_names = ['Model 1', 'Model 2', 'Model 3', 'Model 4 (Full)']

header = f"{'Variable':<30}" + "".join(f"{n:>18}" for n in prog_names)
print(f"\n{header}")
print("─" * (30 + 18 * len(prog_names)))

for var, label in zip(all_vars, all_labels):
    # Coefficient row
    vals = []
    for m in prog_models:
        if var in m.params.index:
            b = m.params[var]
            p = m.pvalues[var]
            stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            vals.append(f"{b:.4f}{stars}")
        else:
            vals.append("—")
    print(f"{label:<30}" + "".join(f"{v:>18}" for v in vals))

    # Standard error row
    se_vals = []
    for m in prog_models:
        if var in m.params.index:
            se_vals.append(f"({m.bse[var]:.4f})")
        else:
            se_vals.append("")
    print(f"{'':<30}" + "".join(f"{v:>18}" for v in se_vals))

print("─" * (30 + 18 * len(prog_names)))

# Model stats
for stat_name, stat_fn in [
    ('N', lambda m: f"{int(m.nobs)}"),
    ('R²', lambda m: f"{m.rsquared:.4f}"),
    ('Adj. R²', lambda m: f"{m.rsquared_adj:.4f}"),
    ('F-statistic', lambda m: f"{m.fvalue:.2f}"),
    ('Prob (F)', lambda m: f"{m.f_pvalue:.2e}"),
    ('AIC', lambda m: f"{m.aic:.1f}"),
    ('BIC', lambda m: f"{m.bic:.1f}"),
    ('Durbin-Watson', lambda m: f"{sm.stats.stattools.durbin_watson(m.resid):.3f}"),
]:
    vals = [stat_fn(m) for m in prog_models]
    print(f"{stat_name:<30}" + "".join(f"{v:>18}" for v in vals))

print("─" * (30 + 18 * len(prog_names)))
print("Standard errors in parentheses. *** p<0.001  ** p<0.01  * p<0.05")


# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  INTERPRETATION GUIDE")
print("=" * 70)
print(f"""
The full model (Model 4) explains {m4d.rsquared_adj*100:.1f}% of IP variation (Adj. R²).

Key coefficients from the full model:
  - US Imports:     β = {m4d.params['usa_imp_sa_thousands']:.4f} (p = {m4d.pvalues['usa_imp_sa_thousands']:.4f})
  - Exports/GDP:    β = {m4d.params['exports_gdp_pct']:.4f} (p = {m4d.pvalues['exports_gdp_pct']:.4f})
  - Interaction:    β = {m4d.params['usa_imp_sa_thousands:exports_gdp_pct']:.6f} (p = {m4d.pvalues['usa_imp_sa_thousands:exports_gdp_pct']:.4f})
  - CPI:            β = {m4d.params['cpi_yoy_nsa']:.4f} (p = {m4d.pvalues['cpi_yoy_nsa']:.4f})
  - Exchange Rate:  β = {m4d.params['exchnage_rate_vs_usd']:.4f} (p = {m4d.pvalues['exchnage_rate_vs_usd']:.4f})

Note: The dependent variable is IP in billions (not growth).
      US imports are in thousands.
      The interaction term captures how trade dependency amplifies
      the US demand effect on industrial production levels.
""")

print("✅ All regressions complete.")

✅ Libraries loaded.
Loaded: 912 rows x 11 columns
✅ Loaded: 912 rows × 11 columns
   Columns: ['country', 'countrycode', 'year', 'month', 'cpi_yoy_nsa', 'exchnage_rate_vs_usd', 'ind_prod_const_sa', 'usa_imp_sa', 'exports_gdp_pct', 'gdp_per_capita_ppp', 'population']
✅ Variables prepared:
   Dependent: ind_prod_const_sa_billions (IP in billions)
   Independent:
     - usa_imp_sa_thousands (US Imports in thousands)
     - exports_gdp_pct (Exports/GDP %)
     - usa_imp_sa_thousands:exports_gdp_pct (Interaction)
     - cpi_yoy_nsa (CPI YoY %)
     - exchnage_rate_vs_usd (Exchange Rate)

   Observations: 912
   Countries: ['Malaysia', 'Singapore', 'Philippines', 'Thailand']

  MODEL 1: FULL MULTIVARIATE OLS (Pooled, all countries)
  IP (billions) = α + β₁(US_Imp) + β₂(Exp/GDP) + β₃(US_Imp × Exp/GDP)
                + β₄(CPI) + β₅(Exchange Rate) + ε
                                OLS Regression Results                                
Dep. Variable:     ind_prod_const_sa_billions   R-squared